In [2]:
from torch.nn.functional import dropout
import torch

In [1]:
import torch

/home/hyang/anaconda3/envs/clip/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
a = torch.zeros(10)

In [10]:
a.scatter_(0, torch.tensor(2), 1)

tensor([0., 0., 1., 0., 0., 0., 0., 0., 0., 0.])

In [8]:
torch.load("/home/hyang/HL_Contrast/SpCo/dataset/cora/scope_1.pt").shape

torch.Size([2, 13264])

In [3]:
torch.rand(1)

tensor([0.5236])

In [16]:
z = torch.tensor([[1,2,3,4,5], [1,2,3,4,5]])

In [18]:
drop_feature(z, 0.)

tensor([[1, 2, 3, 4, 5],
        [1, 2, 3, 4, 5]])

In [3]:
import sys
sys.path.append("/home/hyang/HL_Contrast/Non-Homophily-Large-Scale/")
from dataset import load_nc_dataset

python Scale_GRACE.py --config ./configs/fbgcn.json --dataset fb100 --sub_dataset Penn94 --hidden_dim 128 --runs 1 --train_batch cluster --num_parts 100 --cluster_batch_size 5 --preepochs 1

python Scale_BGRL.py --config ./configs/fbgcn.json --dataset fb100 --sub_dataset Penn94 --hidden_dim 128 --runs 1 --train_batch cluster --num_parts 100 --cluster_batch_size 5 --preepochs 1

python Scale_FBGCN.py --config ./configs/fbgcn.json --dataset penn94 --hidden_dim 128 --runs 1 --train_batch cluster --num_parts 200 --cluster_batch_size 1

In [4]:
import argparse
import sys
import os
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.utils import to_undirected, sort_edge_index
from torch_geometric.data import NeighborSampler, ClusterData, ClusterLoader, Data, GraphSAINTNodeSampler, GraphSAINTEdgeSampler, GraphSAINTRandomWalkSampler, RandomNodeSampler
from torch_scatter import scatter
import GCL.losses as L
from torch_geometric.utils import get_laplacian
from torch.optim import Adam
from GCL.eval import get_split
from Evaluator import LREvaluator
from GCL.models import DualBranchContrast
from model.trial_pretrain import GCNConv
from utility.config import get_arguments
from model.pretrain import get_augmentor
import numpy as np
from MIX_FBGCN_AUG import GConv, Encoder


sys.path.append("/home/hyang/HL_Contrast/Non-Homophily-Large-Scale/")
from data_utils import normalize, gen_normalized_adjs, evaluate, eval_acc, eval_rocauc, to_sparse_tensor
from parse import parse_method, parser_add_main_args
from batch_utils import nc_dataset_to_torch_geo, torch_geo_to_nc_dataset, AdjRowLoader, make_loader
from dataset import load_nc_dataset, NCDataset

In [8]:
# dataset.label.flatten()[torch.where(dataset.label.flatten()>= 0)[0]] < 0

tensor([False, False, False,  ..., False, False, False])

In [9]:
# dataset.label.shape[1]

1

In [10]:
# test_loader = make_loader(0, dataset, split_idx['test'], False, device='cpu', test=True)

/home/hyang/anaconda3/envs/torch/lib/python3.8/site-packages/torch_geometric/deprecation.py:13: UserWarning: 'data.RandomNodeSampler' is deprecated, use 'loader.RandomNodeSampler' instead
  warnings.warn(out)


In [15]:
# for tg_batch in test_loader:
#     print(torch.where(tg_batch.y>-1)[0])

tensor([    0,     1,     2,  ..., 41551, 41552, 41553])


In [12]:
# tg_batch.y[22]

tensor([-1])

In [35]:
dataset = load_nc_dataset('genius')
if len(dataset.label.shape) == 1:
    dataset.label = dataset.label.unsqueeze(1)

split_idx = dataset.get_idx_split(train_prop=0.1, valid_prop=0.1)
# train_idx = split_idx['train']
# train_idx = train_idx.to(device)

n = dataset.graph['num_nodes']
# infer the number of classes for non one-hot and one-hot labels
c = max(dataset.label.max().item() + 1, dataset.label.shape[1])
d = dataset.graph['node_feat'].shape[1]

edge = dataset.graph['edge_index']
y = dataset.graph

In [1]:
from torch_geometric.datasets import Planetoid, WebKB, WikipediaNetwork
from torch_geometric import transforms as T
file_loc = './data/'
dataset_name = "texas"
# dataset = Planetoid(root=file_loc+dataset_name, name=dataset_name, transform=T.NormalizeFeatures())
dataset = WebKB(root=file_loc+dataset_name, name=dataset_name, transform=T.NormalizeFeatures())
# dataset = WikipediaNetwork(root=file_loc+dataset_name, name=dataset_name, transform=T.NormalizeFeatures())
data = dataset[0]
# k = homophily((edge), dataset.label, None, 'node').detach().cpu().numpy()
# k.mean()

/home/hyang/anaconda3/envs/clip/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
a = data.edge_index

In [4]:
import torch

In [8]:
b = torch.cat((a[0], a[1])).unique()

In [10]:
torch.ones(a.shape[1])

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 

/home/hyang/anaconda3/envs/clip/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tensor([[0.1111, 0.0048, 0.0058,  ..., 0.0062, 0.0159, 0.0171],
        [0.0048, 0.0435, 0.0069,  ..., 0.0048, 0.0031, 0.0033],
        [0.0058, 0.0069, 0.0526,  ..., 0.0058, 0.0038, 0.0040],
        ...,
        [0.0062, 0.0048, 0.0058,  ..., 0.0556, 0.0000, 0.0043],
        [0.0159, 0.0031, 0.0038,  ..., 0.0000, 0.0714, 0.0110],
        [0.0171, 0.0033, 0.0040,  ..., 0.0043, 0.0110, 0.0769]])

In [ ]:
sum(dataset[0].test_mask.T[0])/len(dataset[0].train_mask.T[0])

In [20]:
k = torch.tensor([-1,1,2,3])
torch.topk(k, 2)[0]

tensor([3, 2])

In [23]:
torch.topk(-k, 2)[0]

tensor([ 1, -1])

START

In [5]:
# def CCNS(data, sample_num):
#     num_class = data.y.unique().size()[0]
#     ccns_sim = torch.zeros(num_class, num_class)
#     for i in range(num_class):
#         for j in range(num_class):
            

In [3]:
from torch_geometric.nn import MessagePassing
from torch.nn import Linear, Parameter
import torch
from torch import Tensor
from torch_geometric.utils import get_laplacian,add_self_loops, degree
from torch_geometric.typing import Adj, OptTensor, SparseTensor
import os.path as osp
import GCL.losses as L
import GCL.augmentors as A
import torch.nn.functional as F
import torch_geometric.transforms as T
from tqdm import tqdm
from torch.optim import Adam
from GCL.eval import get_split
from Evaluator import LREvaluator
from GCL.models import DualBranchContrast
import numpy as np
from torch_geometric.nn import GCNConv
from GCL.losses import Loss
from abc import ABC, abstractmethod
from torch_geometric.datasets import Planetoid, WebKB, WikipediaNetwork, Actor
from utility.data import HeterophilousGraphDataset, dataset_split
import argparse
from torch_geometric.utils import scatter
class High_Pass(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')  # "Add" aggregation (Step 5).
        self.lin = Linear(in_channels, out_channels, bias=False)
        self.bias = Parameter(torch.Tensor(out_channels))

        self.reset_parameters()

    def reset_parameters(self):
        self.lin.reset_parameters()
        self.bias.data.zero_()

    def forward(self, x, edge_index, edge_weight):
        # x has shape [N, in_channels]
        # edge_index has shape [2, E]

        # Step 1: Add self-loops to the adjacency matrix.
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        edge_index, edge_weight = get_laplacian(edge_index, edge_weight, normalization="sym")

        # Step 2: Linearly transform node feature matrix.
        x = self.lin(x)

        # Step 3: Compute normalization.
        # row, col = edge_index
        # deg = degree(col, x.size(0), dtype=x.dtype)
        # deg_inv_sqrt = deg.pow(-0.5)
        # deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
        # norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        # Step 4-5: Start propagating messages.
        out = self.propagate(edge_index, x=x, edge_weight=edge_weight)

        # Step 6: Apply a final bias vector.
        out += self.bias

        return out

    def message(self, x_j, edge_weight):
        # x_j has shape [E, out_channels]

        # Step 4: Normalize node features.
        return edge_weight.view(-1, 1) * x_j

class Low_Pass(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')  # "Add" aggregation (Step 5).
        self.lin = Linear(in_channels, out_channels, bias=False)
        self.bias = Parameter(torch.Tensor(out_channels))

        self.reset_parameters()

    def reset_parameters(self):
        self.lin.reset_parameters()
        self.bias.data.zero_()

    def forward(self, x, edge_index, edge_weight):
        # x has shape [N, in_channels]
        # edge_index has shape [2, E]

        # Step 1: Add self-loops to the adjacency matrix.
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        # edge_index, edge_weight = get_laplacian(edge_index, edge_weight, normalization="sym")

        # Step 2: Linearly transform node feature matrix.
        x = self.lin(x)

        # Step 3: Compute normalization.
        row, col = edge_index
        deg = degree(col, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        # Step 4-5: Start propagating messages.
        out = self.propagate(edge_index, x=x, norm=norm)

        # Step 6: Apply a final bias vector.
        out += self.bias

        return out

    def message(self, x_j, norm):
        # x_j has shape [E, out_channels]

        # Step 4: Normalize node features.
        return norm.view(-1, 1) * x_j
class Mix_Pass(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')  # "Add" aggregation (Step 5).
        self.lin = Linear(in_channels, out_channels, bias=False)
        self.bias = Parameter(torch.Tensor(out_channels))

        self.reset_parameters()

    def reset_parameters(self):
        self.lin.reset_parameters()
        self.bias.data.zero_()
    def forward(self, x, edge_index, edge_weight, high_pass = False):
        if high_pass:
            edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
            edge_index, edge_weight = get_laplacian(edge_index, edge_weight, normalization="sym")
            
            x = self.lin(x)
            out = self.propagate(edge_index, x=x, edge_weight=edge_weight, high_pass=high_pass)
            out += self.bias
        else:
            edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
            # edge_index, edge_weight = get_laplacian(edge_index, edge_weight, normalization="sym")

            # Step 2: Linearly transform node feature matrix.
            x = self.lin(x)

            # Step 3: Compute normalization.
            row, col = edge_index
            deg = degree(col, x.size(0), dtype=x.dtype)
            deg_inv_sqrt = deg.pow(-0.5)
            deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
            norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

            # Step 4-5: Start propagating messages.
            out = self.propagate(edge_index, x=x, norm=norm, high_pass=high_pass)

            # Step 6: Apply a final bias vector.
            out += self.bias
        return out
    def message(self, x_j, high_pass, norm=None, edge_weight=None):
        if high_pass:
            return edge_weight.view(-1, 1) * x_j
        else:
            return norm.view(-1, 1) * x_j

In [10]:
import torch
torch.set_printoptions(20)
a = torch.load("all_save_1.pt")
b = torch.load("all_save_2.pt")
torch.cosine_similarity(a[1], b[1], dim=1).sum() + (torch.cosine_similarity(a[1], b[1], dim=1) < 1e-4 ).sum()

tensor(2174.17993164062500000000, device='cuda:1', grad_fn=<AddBackward0>)

In [11]:
import torch
torch.set_printoptions(20)
a = torch.load("all_save_SGD_1.pt")
b = torch.load("all_save_SGD_2.pt")
torch.cosine_similarity(a[1], b[1], dim=1).sum() + (torch.cosine_similarity(a[1], b[1], dim=1) < 1e-4 ).sum()

tensor(2277.00390625000000000000, device='cuda:1', grad_fn=<AddBackward0>)

In [194]:
a[0][3].shape

torch.Size([2277, 4554])

In [187]:
print(torch.cosine_similarity(a[0][4], b[0][4], dim=0))

tensor(0., device='cuda:1', grad_fn=<SumBackward1>)


In [184]:
b[0][4]

tensor(-0., device='cuda:1', requires_grad=True)

In [151]:
sim = F.normalize(a[0][0]) @ F.normalize(a[0][1]).t()

In [161]:
exp_sim = torch.exp(sim) * (a[0][2] + a[0][3])

In [169]:
log_prob = sim - torch.log(exp_sim.sum(dim=1,keepdim=True))

In [172]:
loss = log_prob * a[0][2]

In [176]:
(loss.sum(dim=1)/a[0][2].sum(dim=1)).mean()

tensor(1.4868440701e-08, device='cuda:1', grad_fn=<MeanBackward0>)

In [25]:
import torch.nn.functional as F
from torch_geometric.utils import to_undirected, degree
import copy

def KNN_graph(x, k=12, inverse=False):
    # KNN-graph
    h = F.normalize(x, dim=-1)
    device = x.device
    logits = torch.matmul(h, h.t())
    if inverse:
        _, indices = torch.topk(-logits, k=k, dim=-1)
    else:
        _, indices = torch.topk(logits, k=k, dim=-1)
    graph = torch.zeros(h.shape[0], h.shape[0], dtype=torch.int64, device=device).scatter_(1, indices, 1)
    
    edge_index = torch.nonzero(graph).t()
    edge_index = to_undirected(edge_index)
    
    return edge_index


In [45]:
from torch_geometric.datasets import Planetoid, WebKB, WikipediaNetwork, Actor, HeterophilousGraphDataset
from torch_geometric import transforms as T
from torch_geometric.utils import get_laplacian,add_self_loops
import torch
import random
from GCL.eval import get_split
from torch_geometric.utils import homophily
from torch_geometric.utils import (
    coalesce,
    remove_self_loops,
    to_edge_index,
    to_torch_csr_tensor,
)
file_loc = './data/'
dataset_name = "texas"
# dataset = HeterophilousGraphDataset(root=file_loc+dataset_name, name=dataset_name, transform=T.NormalizeFeatures())
# dataset=Actor(root=file_loc+dataset_name,transform=T.NormalizeFeatures())
# dataset = Planetoid(root=file_loc+dataset_name, name=dataset_name, transform=T.NormalizeFeatures())
dataset = WebKB(root=file_loc+dataset_name, name=dataset_name, transform=T.NormalizeFeatures())
# dataset = WikipediaNetwork(root=file_loc+dataset_name, name=dataset_name, transform=T.NormalizeFeatures())
data = dataset[0]
mlp_model = torch.nn.Sequential(
    torch.nn.Linear(data.x.shape[1], data.x.shape[1]),
    torch.nn.ReLU(),
    torch.nn.Linear(data.x.shape[1], data.x.shape[1]),
    torch.nn.ReLU())
neigh_num = 10
high_model = High_Pass(data.x.shape[1], data.x.shape[1])
high_x = high_model(data.x, data.edge_index, data.edge_weight)
high_edge_index = KNN_graph(high_x, k=neigh_num)
a = homophily(high_edge_index, data.y, method="edge_insensitive")

low_model = Low_Pass(data.x.shape[1], data.x.shape[1])
low_x = low_model(data.x, data.edge_index, data.edge_weight)
low_edge_index = KNN_graph(low_x, k=neigh_num)
b = homophily(low_edge_index, data.y, method="edge_insensitive")
union_edge = torch.concat((high_edge_index.T, low_edge_index.T)).unique(dim=0).T
c = homophily(union_edge, data.y, method="edge_insensitive")
combined = torch.cat((high_edge_index.T, low_edge_index.T))
uniques, counts = combined.unique(return_counts=True, dim=0)
difference = uniques[counts == 1]
intersection = uniques[counts > 1]
intersect_edge = intersection.T
d = homophily(intersect_edge, data.y, method="edge_insensitive")
origin_edge = KNN_graph(data.x, k=neigh_num)
e = homophily(origin_edge, data.y, method="edge_insensitive")
mlp_x = mlp_model(data.x)
mlp_edge = KNN_graph(mlp_x, k=neigh_num)
f = homophily(mlp_edge, data.y, method="edge_insensitive")
g = homophily(data.edge_index, data.y, method="edge_insensitive")

In [46]:
print("high pass only: {}".format(a))
print("low pass only: {}".format(b))
print("union edges: {}".format(c))
print("intersection edges: {}".format(d))
print("origin feature: {}".format(e))
print("mlp feature: {}".format(f))
print("initial homo: {}".format(g))

high pass only: 0.2734626531600952
low pass only: 0.09607303142547607
union edges: 0.13457560539245605
intersection edges: 0.4746010899543762
origin feature: 0.26883363723754883
mlp feature: 0.2129422277212143
initial homo: 0.0


In [47]:
high_edge_index_dissim = KNN_graph(high_x, k=neigh_num, inverse=True)
low_edge_index_dissim = KNN_graph(low_x, k=neigh_num, inverse=True)
h = homophily(high_edge_index_dissim, data.y, method="edge_insensitive")
i = homophily(low_edge_index_dissim, data.y, method="edge_insensitive")
combined = torch.cat((high_edge_index_dissim.T, low_edge_index_dissim.T))
uniques, counts = combined.unique(return_counts=True, dim=0)
difference = uniques[counts == 1]
intersection = uniques[counts > 1]
intersect_edge = intersection.T
union_edge = torch.concat((high_edge_index_dissim.T, low_edge_index_dissim.T)).unique(dim=0).T
j = homophily(intersect_edge, data.y, method="edge_insensitive")
k = homophily(union_edge, data.y, method="edge_insensitive")

In [48]:
print(h)
print(i)
print(j)
print(k)

0.0008767284452915192
0.000740978866815567
0.013741564005613327
0.0


In [12]:

low_low_other = []
high_high_other = []
low_high_other = []
high_mlp_other = 0
low_mlp_other = 0
low_low_us = []
high_high_us = []
low_high_us = []
high_mlp_us = 0
low_mlp_us = 0
for data_point in range(data.y.shape[0]):
    for i in data.edge_index[1][data.edge_index[0] == data_point]:
        if data.y[i] == data.y[data_point]:
            low_low_us.append(F.cosine_similarity(low_x[data_point], low_x[i], dim=0))
            high_high_us.append(F.cosine_similarity(high_x[data_point], high_x[i], dim=0))
            low_high_us.append(F.cosine_similarity(low_x[data_point], high_x[i], dim=0))
            # high_mlp_us = high_mlp_us + F.cosine_similarity(high_x[data_point], mlp_x[i], dim=0)
            # low_mlp_us = low_mlp_us + F.cosine_similarity(low_x[data_point], mlp_x[i], dim=0)
        else:
            low_low_other.append(F.cosine_similarity(low_x[data_point], low_x[i], dim=0))
            high_high_other.append(F.cosine_similarity(high_x[data_point], high_x[i], dim=0))
            low_high_other.append(F.cosine_similarity(low_x[data_point], high_x[i], dim=0))
            # high_mlp_other = high_mlp_other + F.cosine_similarity(high_x[data_point], mlp_x[i], dim=0)
            # low_mlp_other = low_mlp_other + F.cosine_similarity(low_x[data_point], mlp_x[i], dim=0)
# print("low_low_us:{}, low_low_other:{}, diff: {}".format(low_low_us, low_low_other/other, low_low_us - low_low_other/other))
# print("high_high_us:{}, high_high_other:{}, diff: {}".format(high_high_us, high_high_other/other, high_high_us- high_high_other/other))
# print("low_high_us:{}, low_high_other:{}, diff: {}".format(low_high_us, low_high_other/other, low_high_us-low_high_other/other))
# print("high_mlp_us:{}, high_mlp_other:{}, diff: {}".format(high_mlp_us, high_mlp_other/other, high_mlp_us- high_mlp_other/other))
# print("low_mlp_us:{}, low_mlp_other:{}, diff: {}".format(low_mlp_us, low_mlp_other/other, low_mlp_us- low_mlp_other/other))


In [17]:
torch.std(torch.stack(low_low_us))

tensor(0.3152, grad_fn=<StdBackward0>)

In [18]:
torch.std(torch.stack(low_low_other))

tensor(0.2985, grad_fn=<StdBackward0>)

In [ ]:
F.normalize(torch.tensor([1.,1.])) @ F.normalize(torch.tensor([1., 0.]))

In [148]:
low_low_us.std

<function Tensor.std>

In [142]:
print("low_low_us:{}, low_low_other:{}, diff: {}".format(low_low_us/us, low_low_other/other, (low_low_us/us - low_low_other/other)))
print("high_high_us:{}, high_high_other:{}, diff: {}".format(high_high_us/us, high_high_other/other, high_high_us/us- high_high_other/other))

low_low_us:0.40469345450401306, low_low_other:0.3779071867465973, diff: 0.02678626775741577
high_high_us:-0.06028575077652931, high_high_other:-0.07070158421993256, diff: 0.010415833443403244
low_high_us:-0.0007861055782996118, low_high_other:-0.00010549993749009445, diff: -0.000680605648085475
high_mlp_us:0.00021747600112576038, high_mlp_other:9.648747072787955e-05, diff: 0.00012098853039788082
low_mlp_us:-0.006466635502874851, low_mlp_other:-0.004971200600266457, diff: -0.0014954349026083946


In [141]:
low_low_us/us - low_low_other/other

tensor(0.0268, grad_fn=<SubBackward0>)

In [ ]:
low_low_us/us - low_low_other/other

high_high_us:0.11244041472673416, high_high_other:0.04693984240293503


In [72]:
F.cosine_similarity(data.x, high_x).max()

tensor(0.0948, grad_fn=<MaxBackward1>)

In [65]:
# for neighbor in neighbors:
F.cosine_similarity(mlp_x[0], low_x[0], dim=0)
# diff_high = F.cosine_similarity(mlp_x[0], high_x[0])

tensor(-0.0070, grad_fn=<SumBackward1>)

In [25]:
device = torch.device("cuda:{}".format(0))
split = get_split(data.x.size()[0], train_ratio=0.1, test_ratio=0.8)
split["train"] = split["train"].to(device)
neg_sample=[]
N = data.num_nodes
adj = to_torch_csr_tensor(data.edge_index, size=(N, N))
edge_index2, _ = to_edge_index(adj @ adj)
edge_index2, _ = remove_self_loops(edge_index2)
edge_index = torch.cat([data.edge_index, edge_index2], dim=1)
for j in range(len(data.y)):
    if j in split["train"]:
        neighbors = torch.where(edge_index[0] == j)[0]
        irrelevant_neighbors = edge_index[1][neighbors][data.y[edge_index[1][neighbors]] != data.y[j]]
        irrelevant_neighbors = torch.tensor([i for i in irrelevant_neighbors if i in split["train"]])
        neg_mask = torch.zeros(data.x.shape[0])
        if len(irrelevant_neighbors) > 0:
            neg_mask[irrelevant_neighbors] = 1.
        neg_sample.append(neg_mask)
    else:
        neg_sample.append(torch.zeros(data.x.shape[0]))

In [6]:
[i for i in irrelevant_neighbors if i not in [476]]

[tensor(2545)]

In [35]:
irrelevant_neighbors

tensor([  2, 652, 654])

tensor(3)

tensor([], dtype=torch.int64)

In [52]:
(intersect_edge).shape[1]/(low_edge_index).shape[1]

0.3450676108142905

In [4]:
from GCL.eval import get_split
split = get_split(data.x.size()[0], train_ratio=0.1, test_ratio=0.8)

In [9]:
a = torch.tensor([0])

In [ ]:
torch.tensor(range(data.num_nodes)) in split["train"]

tensor([False, False, False,  ..., False, False, False])

tensor([  12,   13,   17,   43,   66,   70,   72,  104,  106,  122,  127,  129,
         130,  132,  151,  152,  167,  177,  199,  236,  240,  259,  266,  269,
         287,  296,  311,  337,  355,  357,  367,  369,  386,  407,  417,  421,
         517,  556,  558,  578,  580,  597,  626,  648,  699,  720,  724,  729,
         744,  746,  747,  765,  779,  783,  795,  798,  801,  808,  834,  854,
         860,  862,  867,  871,  874,  890,  892,  896,  899,  900,  908,  946,
         956,  980,  988,  997, 1005, 1017, 1018, 1021, 1028, 1029, 1032, 1089,
        1091, 1092, 1128, 1135, 1176, 1182, 1183, 1210, 1215, 1216, 1230, 1236,
        1244, 1251, 1260, 1261, 1264, 1266, 1273, 1281, 1298, 1324, 1325, 1326,
        1343, 1347, 1349, 1358, 1366, 1379, 1411, 1417, 1421, 1433, 1451, 1455,
        1458, 1459, 1470, 1493, 1494, 1495, 1496, 1500, 1505, 1506, 1522, 1529,
        1557, 1563, 1568, 1586, 1589, 1603, 1604, 1607, 1615, 1629, 1640, 1647,
        1681, 1706, 1716, 1719, 1762, 17

In [28]:
neg_masks = []
for i in range(torch.unique(data.y).shape[0]):
    nongroup = torch.where(data.y!=i)[0]
    neg_mask = torch.zeros(data.x.shape[0])
    neg_mask[nongroup] = 1.
    neg_masks.append(neg_mask)
neg_sample=[]
for j in range(len(data.y)):
    if j in split["train"]:
        neg_sample.append(neg_masks[data.y[j]])
    else:
        neg_sample.append(torch.zeros(data.x.shape[0]))
neg_sample = torch.stack(neg_sample)

In [30]:
neg_sample

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [1., 0., 0.,  ..., 1., 1., 1.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [12]:
neg_sample.shape

torch.Size([2708, 2708])

In [27]:
import torch
import os.path as osp
import GCL.losses as L
import GCL.augmentors as A
import torch.nn.functional as F
import torch_geometric.transforms as T
from tqdm import tqdm
from torch.optim import Adam
from GCL.eval import get_split
from Evaluator import LREvaluator
from GCL.models import DualBranchContrast
import numpy as np
from torch_geometric.nn import GCNConv
class GConv(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, activation, num_layers):
        super(GConv, self).__init__()
        self.activation = activation()
        self.layers = torch.nn.ModuleList()
        self.layers.append(GCNConv(input_dim, hidden_dim, cached=False))
        for _ in range(num_layers - 1):
            self.layers.append(GCNConv(hidden_dim, hidden_dim, cached=False))

    def forward(self, x, edge_index, edge_weight=None):
        z = x
        for i, conv in enumerate(self.layers):
            z = conv(z, edge_index, edge_weight)
            z = self.activation(z)
        return z
class Encoder(torch.nn.Module):
    def __init__(self, encoder, augmentor, hidden_dim, proj_dim):
        super(Encoder, self).__init__()
        self.encoder = encoder 
        self.augmentor = augmentor

        self.fc1 = torch.nn.Linear(hidden_dim, proj_dim)
        self.fc2 = torch.nn.Linear(proj_dim, hidden_dim)

    def forward(self, x, edge_index, edge_weight=None):
        aug1, aug2 = self.augmentor
        x1, edge_index1, edge_weight1 = aug1(x, edge_index, edge_weight)
        x2, edge_index2, edge_weight2 = aug2(x, edge_index, edge_weight)
        z = self.encoder(x, edge_index, edge_weight)
        z1 = self.encoder(x1, edge_index1, edge_weight1)
        z2 = self.encoder(x2, edge_index2, edge_weight2)
        return z, z1, z2

    def project(self, z: torch.Tensor) -> torch.Tensor:
        z = F.elu(self.fc1(z))
        return self.fc2(z)
def train(encoder_model, contrast_model, x, edge_index, edge_weight, optimizer, neg_mask = None):
    encoder_model.train()
    optimizer.zero_grad()
    z, z1, z2 = encoder_model(x, edge_index, edge_weight)
    h1, h2 = [encoder_model.project(x) for x in [z1, z2]]
    loss = contrast_model(h1, h2, neg_mask1 = neg_mask, neg_mask2 = neg_mask)
    loss.backward()
    optimizer.step()
    return loss.item()


def test(encoder_model, x, edge_index, edge_weight,y):
    encoder_model.eval()
    z, _, _ = encoder_model(x, edge_index, edge_weight)
    split = get_split(num_samples=z.size()[0], train_ratio=0.1, test_ratio=0.8)
    result = LREvaluator()(z, y, split)
    return result

In [28]:
from torch_geometric.utils import get_num_hops

In [43]:
model = GConv(data.x.shape[1], 512, torch.nn.ReLU, 3)
get_num_hops(model)

3

In [44]:
model(data.x, data.edge_index)

tensor([[0.0000, 0.0004, 0.0018,  ..., 0.0001, 0.0000, 0.0000],
        [0.0013, 0.0011, 0.0014,  ..., 0.0000, 0.0000, 0.0023],
        [0.0001, 0.0000, 0.0017,  ..., 0.0000, 0.0000, 0.0009],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0027,  ..., 0.0010, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0023,  ..., 0.0008, 0.0000, 0.0000]],
       grad_fn=<ReluBackward0>)

In [13]:
from GCL.losses import Loss
from abc import ABC, abstractmethod
class Sampler(ABC):
    def __init__(self, intraview_negs=False):
        self.intraview_negs = intraview_negs

    def __call__(self, anchor, sample, *args, **kwargs):
        ret = self.sample(anchor, sample, *args, **kwargs)
        if self.intraview_negs:
            ret = self.add_intraview_negs(*ret)
        return ret
    
    @abstractmethod
    def sample(self, anchor, sample, *args, **kwargs):
        pass

    @staticmethod
    def add_intraview_negs(anchor, sample, pos_mask, neg_mask):
        num_nodes = anchor.size(0)
        device = anchor.device
        intraview_pos_mask = torch.zeros_like(pos_mask, device=device)
        intraview_neg_mask = torch.ones_like(pos_mask, device=device) - torch.eye(num_nodes, device=device)
        new_sample = torch.cat([sample, anchor], dim=0)                     # (M+N) * K
        new_pos_mask = torch.cat([pos_mask, intraview_pos_mask], dim=1)     # M * (M+N)
        new_neg_mask = torch.cat([neg_mask, intraview_neg_mask], dim=1)     # M * (M+N)
        return anchor, new_sample, new_pos_mask, new_neg_mask
    
class SameScaleSampler(Sampler):
    def __init__(self, *args, **kwargs):
        super(SameScaleSampler, self).__init__(*args, **kwargs)

    def sample(self, anchor, sample, *args, **kwargs):
        assert anchor.size(0) == sample.size(0)
        num_nodes = anchor.size(0)
        device = anchor.device
        pos_mask = torch.eye(num_nodes, dtype=torch.float32, device=device)
        neg_mask = 1. - pos_mask
        return anchor, sample, pos_mask, neg_mask
        # return anchor, sample, pos_mask
    
def get_sampler(mode: str, intraview_negs: bool) -> Sampler:
    return SameScaleSampler(intraview_negs=intraview_negs)
    
class DualBranchContrast(torch.nn.Module):
    def __init__(self, loss: Loss, mode: str, intraview_negs: bool = False, **kwargs):
        super(DualBranchContrast, self).__init__()
        self.loss = loss
        self.mode = mode
        self.sampler = get_sampler(mode, intraview_negs=intraview_negs)
        self.kwargs = kwargs

    def forward(self, h1=None, h2=None, g1=None, g2=None, batch=None, h3=None, h4=None,
                extra_pos_mask=None, extra_neg_mask=None, neg_mask1 = None, neg_mask2 = None):
        if self.mode == 'L2L':
            if neg_mask1 is None:
                print("No neg mask")
                anchor1, sample1, pos_mask1, neg_mask1 = self.sampler(anchor=h1, sample=h2)
                anchor2, sample2, pos_mask2, neg_mask2 = self.sampler(anchor=h2, sample=h1)
            else:
                anchor1, sample1, pos_mask1, _= self.sampler(anchor=h1, sample=h2)
                anchor2, sample2, pos_mask2, _= self.sampler(anchor=h2, sample=h1)
        # elif self.mode == 'G2G':
        #     assert g1 is not None and g2 is not None
        #     anchor1, sample1, pos_mask1, neg_mask1 = self.sampler(anchor=g1, sample=g2)
        #     anchor2, sample2, pos_mask2, neg_mask2 = self.sampler(anchor=g2, sample=g1)
        # else:  # global-to-local
        #     if batch is None or batch.max().item() + 1 <= 1:  # single graph
        #         assert all(v is not None for v in [h1, h2, g1, g2, h3, h4])
        #         anchor1, sample1, pos_mask1, neg_mask1 = self.sampler(anchor=g1, sample=h2, neg_sample=h4)
        #         anchor2, sample2, pos_mask2, neg_mask2 = self.sampler(anchor=g2, sample=h1, neg_sample=h3)
        #     else:  # multiple graphs
        #         assert all(v is not None for v in [h1, h2, g1, g2, batch])
        #         anchor1, sample1, pos_mask1, neg_mask1 = self.sampler(anchor=g1, sample=h2, batch=batch)
        #         anchor2, sample2, pos_mask2, neg_mask2 = self.sampler(anchor=g2, sample=h1, batch=batch)

        # pos_mask1, neg_mask1 = add_extra_mask(pos_mask1, neg_mask1, extra_pos_mask, extra_neg_mask)
        # pos_mask2, neg_mask2 = add_extra_mask(pos_mask2, neg_mask2, extra_pos_mask, extra_neg_mask)
        l1 = self.loss(anchor=anchor1, sample=sample1, pos_mask=pos_mask1, neg_mask=neg_mask1, **self.kwargs)
        l2 = self.loss(anchor=anchor2, sample=sample2, pos_mask=pos_mask2, neg_mask=neg_mask2, **self.kwargs)

        return (l1 + l2) * 0.5

In [ ]:
hidden_dim=256
proj_dim=128
total_result = []
torch.manual_seed(42)
torch.cuda.manual_seed(42)
device = torch.device("cuda:1")
data = data.to(device)
# intersect_edge = intersect_edge.to(device)
hidden_dim = hidden_dim
pre_learning_rate = 0.0005
preepochs=200
neg_masks = []
for i in range(torch.unique(data.y).shape[0]):
    nongroup = torch.where(data.y!=i)[0]
    neg_mask = torch.zeros(data.x.shape[0])
    neg_mask[nongroup] = 1.
    neg_masks.append(neg_mask)
neg_sample = [neg_masks[i] for i in data.y]
neg_sample = torch.stack(neg_sample)
for i in range(1):
    aug1 = A.Compose([A.EdgeRemoving(pe=0.3), A.FeatureMasking(pf=0.3)])
    aug2 = A.Compose([A.EdgeRemoving(pe=0.3), A.FeatureMasking(pf=0.3)])
    gconv = GConv(input_dim=data.num_features, hidden_dim=hidden_dim, activation=torch.nn.ReLU, num_layers=2).to(device)
    encoder_model = Encoder(encoder=gconv, augmentor=(aug1, aug2), hidden_dim=hidden_dim, proj_dim=hidden_dim).to(device)
    contrast_model = DualBranchContrast(loss=L.InfoNCE(tau=0.2), mode='L2L', intraview_negs=True).to(device)

    optimizer = Adam(encoder_model.parameters(), lr=pre_learning_rate)

    with tqdm(total=preepochs, desc='(T)') as pbar:
        for epoch in range(preepochs):
            loss = train(encoder_model, contrast_model, data.x, data.edge_index, data.edge_weight, optimizer, neg_sample)
            pbar.set_postfix({'loss': loss})
            pbar.update()
    test_result = test(encoder_model, data.x, data.edge_index, data.edge_weight, data.y)
    total_result.append(test_result["accuracy"])

# with open('./results/nc_GRACE_{}_{}.csv'.format(dataset, gnn), 'a+') as file:
#     file.write('\n')
#     file.write('pretrain epochs = {}\n'.format(preepochs))
#     file.write('pre_learning_rate = {}\n'.format(pre_learning_rate))
#     file.write('hidden_dim = {}\n'.format(hidden_dim))
#     file.write('aug = {}\n'.format(aug))
#     file.write('aug2 = {}\n'.format(aug2))
#     file.write('(E): GRACE Mean Accuracy: {}, with Std: {}'.format(np.mean(total_result), np.std(total_result)))

In [46]:
from torch_geometric.nn import MessagePassing
from torch.nn import Linear, Parameter
import torch
from torch import Tensor
from torch_geometric.utils import get_laplacian,add_self_loops, degree
from torch_geometric.typing import Adj, OptTensor, SparseTensor
import os.path as osp
import GCL.losses as L
import GCL.augmentors as A
import torch.nn.functional as F
import torch_geometric.transforms as T
from tqdm import tqdm
from torch.optim import Adam
from GCL.eval import get_split
from Evaluator import LREvaluator
from GCL.models import DualBranchContrast
import numpy as np
from torch_geometric.nn import GCNConv
from GCL.losses import Loss
from abc import ABC, abstractmethod
from torch_geometric.datasets import Planetoid, WebKB, WikipediaNetwork, Actor
from utility.data import HeterophilousGraphDataset, dataset_split
import argparse
from torch_geometric.utils import scatter
class High_Pass(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')  # "Add" aggregation (Step 5).
        self.lin = Linear(in_channels, out_channels, bias=False)
        self.bias = Parameter(torch.Tensor(out_channels))

        self.reset_parameters()

    def reset_parameters(self):
        self.lin.reset_parameters()
        self.bias.data.zero_()

    def forward(self, x, edge_index, edge_weight):
        # x has shape [N, in_channels]
        # edge_index has shape [2, E]

        # Step 1: Add self-loops to the adjacency matrix.
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        edge_index, edge_weight = get_laplacian(edge_index, edge_weight, normalization="sym")

        # Step 2: Linearly transform node feature matrix.
        x = self.lin(x)

        # Step 3: Compute normalization.
        # row, col = edge_index
        # deg = degree(col, x.size(0), dtype=x.dtype)
        # deg_inv_sqrt = deg.pow(-0.5)
        # deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
        # norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        # Step 4-5: Start propagating messages.
        out = self.propagate(edge_index, x=x, edge_weight=edge_weight)

        # Step 6: Apply a final bias vector.
        out += self.bias

        return out

    def message(self, x_j, edge_weight):
        # x_j has shape [E, out_channels]

        # Step 4: Normalize node features.
        return edge_weight.view(-1, 1) * x_j

class Low_Pass(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')  # "Add" aggregation (Step 5).
        self.lin = Linear(in_channels, out_channels, bias=False)
        self.bias = Parameter(torch.Tensor(out_channels))

        self.reset_parameters()

    def reset_parameters(self):
        self.lin.reset_parameters()
        self.bias.data.zero_()

    def forward(self, x, edge_index, edge_weight):
        # x has shape [N, in_channels]
        # edge_index has shape [2, E]

        # Step 1: Add self-loops to the adjacency matrix.
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        # edge_index, edge_weight = get_laplacian(edge_index, edge_weight, normalization="sym")

        # Step 2: Linearly transform node feature matrix.
        x = self.lin(x)

        # Step 3: Compute normalization.
        row, col = edge_index
        deg = degree(col, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        # Step 4-5: Start propagating messages.
        out = self.propagate(edge_index, x=x, norm=norm)

        # Step 6: Apply a final bias vector.
        out += self.bias

        return out

    def message(self, x_j, norm):
        # x_j has shape [E, out_channels]

        # Step 4: Normalize node features.
        return norm.view(-1, 1) * x_j
class Mix_Pass(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')  # "Add" aggregation (Step 5).
        self.lin = Linear(in_channels, out_channels, bias=False)
        self.bias = Parameter(torch.Tensor(out_channels))

        self.reset_parameters()

    def reset_parameters(self):
        self.lin.reset_parameters()
        self.bias.data.zero_()
    def forward(self, x, edge_index, edge_weight, high_pass = False):
        if high_pass:
            edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
            edge_index, edge_weight = get_laplacian(edge_index, edge_weight, normalization="sym")
            
            x = self.lin(x)
            out = self.propagate(edge_index, x=x, edge_weight=edge_weight, high_pass=high_pass)
            out += self.bias
        else:
            edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
            # edge_index, edge_weight = get_laplacian(edge_index, edge_weight, normalization="sym")

            # Step 2: Linearly transform node feature matrix.
            x = self.lin(x)

            # Step 3: Compute normalization.
            row, col = edge_index
            deg = degree(col, x.size(0), dtype=x.dtype)
            deg_inv_sqrt = deg.pow(-0.5)
            deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
            norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

            # Step 4-5: Start propagating messages.
            out = self.propagate(edge_index, x=x, norm=norm, high_pass=high_pass)

            # Step 6: Apply a final bias vector.
            out += self.bias
        return out
    def message(self, x_j, high_pass, norm=None, edge_weight=None):
        if high_pass:
            return edge_weight.view(-1, 1) * x_j
        else:
            return norm.view(-1, 1) * x_j